# 01 — the measurements

Ten relations. Each is computed at run time, tiered against the
generational-lineage operator domain, and labelled `KNOWN` (with attribution) or
`OURS`.

**Exhaustive, not sampled.** R1–R4 and R6–R8 enumerate a finite space completely;
there is no sampling error to report because there is no sampling. R9 counts a
density over a stated range. R10 is the one sampled statistic and it carries a
**stated, uncorrected bias** — see its detail line.

The three kinds of wrong are kept apart, per the skill:

| | |
|---|---|
| `CODE-FAULT` | the check did not run — the claim is **unjudged** |
| `MATHS-FAULT` | both sides measured and they disagree — the claim is **false** |
| method error | correct code, correct maths, wrong question — invisible to both |

**On the record:** the first run of this engine returned three `MATHS-FAULT`s
(R6, R8, R9). All three were errors in *my assertions*, not in the mathematics,
and they are documented inside the engine at the point of each fix rather than
tidied away. R9's correction made the result **better**, not worse — see 03.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
from engine.e_collatz_shift import (
    T, parity_word, parity_vector, branch_affine, rational_cycle,
    CollatzShiftEngine, run,
)

In [2]:
result = run(verbose=True)

COLLATZ AS THE 2-ADIC SHIFT — engine report
10/10 relations hold

relation                      tier  status       provenance
------------------------------------------------------------------------------
collatz.affine_on_classes      t1   HOLDS        KNOWN (folklore)
collatz.parity_bijection       t3   HOLDS        KNOWN (Bernstein 1994; Bernstein-Lagarias 1996)
collatz.shift_conjugacy        t2   HOLDS        KNOWN (Bernstein-Lagarias 1996)
collatz.pascal_refinement      t3   HOLDS        OURS (framing; the maths is elementary once stated)
collatz.gaussian_from_pascal   t3   HOLDS        OURS (framing)
collatz.cycle_denominator      t3   HOLDS        KNOWN (standard; 2^k-3^d=1 only at (2,1) is Levi ben Gerson / Catalan, cited not proved here)
collatz.mod3_orphans           t2   HOLDS        KNOWN (elementary)
collatz.forward_two_to_one     t3   HOLDS        OURS (the measurement; the fact follows from R3)
collatz.backward_deficit       t3   HOLDS        OURS (the measurement)
colla

In [3]:
print(f"\n{result['held']}/{result['total']} relations hold")
assert result['all_hold'], 'a relation did not hold — read the report above'


10/10 relations hold


## R1 by hand — the recursive decomposition, exhibited

`T^k(n) = (3^d n + c) / 2^k` for `n = j (mod 2^k)`, with `d` fixed by the class.

In [4]:
for k in (1, 2, 3):
    print(f'--- k={k}: {1<<k} classes ---')
    for j in range(1 << k):
        d, slope, c = branch_affine(k, j)
        print(f'  n = {j} mod {1<<k}:  T^{k}(n) = (3^{d}·n + {c})/2^{k}'
              f'    slope {slope}')

--- k=1: 2 classes ---
  n = 0 mod 2:  T^1(n) = (3^0·n + 0)/2^1    slope 1/2
  n = 1 mod 2:  T^1(n) = (3^1·n + 1/2)/2^1    slope 3/2
--- k=2: 4 classes ---
  n = 0 mod 4:  T^2(n) = (3^0·n + 0)/2^2    slope 1/4
  n = 1 mod 4:  T^2(n) = (3^1·n + 1/4)/2^2    slope 3/4
  n = 2 mod 4:  T^2(n) = (3^1·n + 1/2)/2^2    slope 3/4
  n = 3 mod 4:  T^2(n) = (3^2·n + 5/4)/2^2    slope 9/4
--- k=3: 8 classes ---
  n = 0 mod 8:  T^3(n) = (3^0·n + 0)/2^3    slope 1/8
  n = 1 mod 8:  T^3(n) = (3^2·n + 7/8)/2^3    slope 9/8
  n = 2 mod 8:  T^3(n) = (3^1·n + 1/4)/2^3    slope 3/8
  n = 3 mod 8:  T^3(n) = (3^2·n + 5/8)/2^3    slope 9/8
  n = 4 mod 8:  T^3(n) = (3^1·n + 1/2)/2^3    slope 3/8
  n = 5 mod 8:  T^3(n) = (3^1·n + 1/8)/2^3    slope 3/8
  n = 6 mod 8:  T^3(n) = (3^2·n + 5/4)/2^3    slope 9/8
  n = 7 mod 8:  T^3(n) = (3^3·n + 19/8)/2^3    slope 27/8


## R4 by hand — the tessellation

Count, over the `2^k` classes, how many odd steps each takes. It is Pascal's
triangle, exactly, with no error term. **This is why the chaos is only apparent:**
the branch structure of the Collatz tree is the most ordered combinatorial object
there is, read through a metric that scrambles it.

In [5]:
import math
for k in range(1, 9):
    hist = {}
    for j in range(1 << k):
        d, _, _ = branch_affine(k, j)
        hist[d] = hist.get(d, 0) + 1
    row = [hist.get(d, 0) for d in range(k + 1)]
    binom = [math.comb(k, d) for d in range(k + 1)]
    print(f'k={k:2d}  measured {str(row):<40} binomial {str(binom):<40} '
          f'{"MATCH" if row == binom else "DIFFER"}')

k= 1  measured [1, 1]                                   binomial [1, 1]                                   MATCH
k= 2  measured [1, 2, 1]                                binomial [1, 2, 1]                                MATCH
k= 3  measured [1, 3, 3, 1]                             binomial [1, 3, 3, 1]                             MATCH
k= 4  measured [1, 4, 6, 4, 1]                          binomial [1, 4, 6, 4, 1]                          MATCH
k= 5  measured [1, 5, 10, 10, 5, 1]                     binomial [1, 5, 10, 10, 5, 1]                     MATCH
k= 6  measured [1, 6, 15, 20, 15, 6, 1]                 binomial [1, 6, 15, 20, 15, 6, 1]                 MATCH
k= 7  measured [1, 7, 21, 35, 35, 21, 7, 1]             binomial [1, 7, 21, 35, 35, 21, 7, 1]             MATCH
k= 8  measured [1, 8, 28, 56, 70, 56, 28, 8, 1]         binomial [1, 8, 28, 56, 70, 56, 28, 8, 1]         MATCH


## R6 by hand — cycles are rationals, and the denominator is `2^k − 3^d`

Every periodic parity word gives exactly one rational cycle. It is an *integer*
cycle only when `2^k − 3^d` divides the numerator, and `2^k − 3^d = 1` happens at
`(k,d) = (2,1)` and nowhere else.

In [6]:
for word in [(1,0), (1,1), (1,0,0), (1,1,0,0,0), (1,1,0,1,0,0,0)]:
    x, d, k, denom = rational_cycle(word)
    print(f"word={''.join(map(str,word)):<10} k={k:2d} d={d:2d} "
          f'2^k-3^d={denom:>7}   x = {x}')

print()
print('the ladder, along the convergents of log2(3) = %.8f:' % math.log(3, 2))
for k, d in ((2,1),(3,2),(5,3),(8,5),(13,8),(19,12),(84,53)):
    print(f'  k={k:3d} d={d:3d}   2^k-3^d = {2**k-3**d:>26}   k/d = {k/d:.8f}')

word=10         k= 2 d= 1 2^k-3^d=      1   x = 1
word=11         k= 2 d= 2 2^k-3^d=     -5   x = -1
word=100        k= 3 d= 1 2^k-3^d=      5   x = 1/5
word=11000      k= 5 d= 2 2^k-3^d=     23   x = 5/23
word=1101000    k= 7 d= 3 2^k-3^d=    101   x = 23/101

the ladder, along the convergents of log2(3) = 1.58496250:
  k=  2 d=  1   2^k-3^d =                          1   k/d = 2.00000000
  k=  3 d=  2   2^k-3^d =                         -1   k/d = 1.50000000
  k=  5 d=  3   2^k-3^d =                          5   k/d = 1.66666667
  k=  8 d=  5   2^k-3^d =                         13   k/d = 1.60000000
  k= 13 d=  8   2^k-3^d =                       1631   k/d = 1.62500000
  k= 19 d= 12   2^k-3^d =                      -7153   k/d = 1.58333333
  k= 84 d= 53   2^k-3^d =   -40432553845953101497907   k/d = 1.58490566


**On "near miss".** This ladder is a *linear form in logarithms* — how close
`k·log2` can come to `d·log3` without equalling it. That is the same **family** as
the Fermat near-miss work in `FourthAgePapers/FermatMonster` and
`VAPMIP/engines/e14_fermat_near_miss.py`: in both cases the question is how
closely towers of small primes can approach one another. It is **not the same
problem**, and it is **not** the Riemann Hypothesis. Recorded as a family
resemblance and nothing more.